# Create mosaic GIF

Assembles one full mosaic (every FOV tiled by stage position) per z-step,
for a chosen **round** and **channel**, into an animated GIF -- watch how
the image content changes across the whole imaged depth for any round/
channel combination, not just the cells/DAPI round `measure_tissue_
thickness_test.ipynb`'s section 24 is built around.

Follows [`NOTEBOOK_GUIDELINES.md`](../../NOTEBOOK_GUIDELINES.md): calculation
cells are cached under `analysis/cache/create_mosaic_gif/` (scoped by round
AND channel, since either can change between runs) and use `ProgressReporter`.

**Cost warning, read before running**: this needs every FOV's raw frame at
*every included z-step* -- reading every FOV at every z-step is heavy,
genuinely multi-hour-scale I/O for a full-resolution sweep across a whole
round. `GIF_Z_STRIDE` subsamples the available z-steps (keeps it to a few
dozen frames, not all of them), and the calculation cell prints a real time
estimate (from this round's HAL `<exposure_time>`) before starting the read
loop. **`USE_SLURM_ARRAY`** (section 4) submits the read as a SLURM array
job instead of running sequentially -- per standing convention, any heavy
multi-FOV I/O task in these notebooks offers this option.


## 1 — Setup

In [ ]:
import os
import sys
import json
import csv
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from skimage.transform import resize as sk_resize

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/misc/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config      import ExperimentConfig
from MERci.common.metadata    import ExperimentMetadata
from MERci.progress           import ProgressTracker
from MERci.progress_display   import ProgressReporter, format_duration
from MERci.common.io          import iter_image_frames
from MERci.analysis.round     import create_mosaic
from MERci.scheduler          import resolve_round_flip_y
from MERci.acquisition.configs import find_frame_table_for_hal_config, read_hal_exposure_time
from MERci.acquisition.merlin_config import load_microscope_orientation, apply_microscope_orientation

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME  = SAMPLE_DIR.name
IMAGE_SUFFIX = ".zarr"   # must match what HAL wrote
MICROSCOPE   = "ST2"     # camera->stage orientation convention for the mosaic (section 3)

# ---- Choose which round and channel to sweep ------------------------------
# By imaging_type (e.g. "cells", "bits"), or set ROUND_ID directly to override.
ROUND_IMAGING_TYPE = "cells"
ROUND_ID            = None

# Wavelength (nm) to render -- must be a real color in the chosen round's frame table.
CHANNEL_NM = 405.0

# ---- GIF parameters --------------------------------------------------------
GIF_Z_STRIDE          = 5      # every Nth z-step -- lower = smoother GIF, more reads
GIF_DOWNSCALE_WIDTH_PX = 1000   # each mosaic frame is downscaled to this width before going into the GIF
GIF_FRAME_DURATION_MS  = 300    # per-frame display duration in the saved GIF

print(f"Sample name : {SAMPLE_NAME}")
print(f"Microscope  : {MICROSCOPE}")
print(f"Round       : imaging_type={ROUND_IMAGING_TYPE!r}  (ROUND_ID override: {ROUND_ID})")
print(f"Channel     : {CHANNEL_NM} nm")

In [ ]:
config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{SAMPLE_NAME}.txt",
    image_suffix   = IMAGE_SUFFIX,
    microscope     = MICROSCOPE,
)

meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)
tracker = ProgressTracker(config.analysis_dir)

figures_dir = config.analysis_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

NOTEBOOK_NAME = "create_mosaic_gif"
cache_dir     = config.analysis_dir / "cache" / NOTEBOOK_NAME

print(f"Rounds : {meta.n_rounds}")
print(f"FOVs   : {meta.n_fovs}")
print(f"Figures: {figures_dir}")
print(f"Cache  : {cache_dir}")

## 3 — Resolve the target round and its frame table

In [ ]:
def resolve_round_id(meta, imaging_type):
    """First round_id whose series carry the given imaging_type."""
    for rid in meta.valid_round_ids():
        if any((s.imaging_type or "").strip().lower() == imaging_type.strip().lower()
               for s in meta.series_for_round(rid)):
            return rid
    raise ValueError(f"No round found with imaging_type={imaging_type!r}")


def load_round_frame_table(round_id, config, meta):
    """Frame table (columns color/channel/z, 0-based frame-index rows) for round_id's HAL config."""
    for s in meta.series_for_round(round_id):
        if not s.hal_config:
            continue
        hal_path = Path(config.settings_dir) / s.hal_config
        ft_path  = find_frame_table_for_hal_config(hal_path, config.metadata_dir)
        if ft_path and ft_path.exists():
            return pd.read_csv(ft_path, index_col=0)
    raise FileNotFoundError(f"No frame table found for round {round_id}")


target_round_id = ROUND_ID if ROUND_ID is not None else resolve_round_id(meta, ROUND_IMAGING_TYPE)
if not meta.round_fully_written(target_round_id):
    print(f"WARNING: round {target_round_id} is not yet fully written on disk -- "
          f"results below will be based on a partial FOV set.")

frame_table = load_round_frame_table(target_round_id, config, meta)

channel_frames = (
    frame_table[frame_table["color"].round(0) == round(CHANNEL_NM)]
    .sort_values("z")
)
if channel_frames.empty:
    available = sorted(frame_table["color"].dropna().unique())
    raise ValueError(f"No frames found for channel {CHANNEL_NM} nm in round {target_round_id}'s "
                      f"frame table. Colors actually present: {available}")

z_frame_indices = list(zip(channel_frames.index.tolist(), channel_frames["z"].tolist()))
z_axis_um       = np.array([z for _, z in z_frame_indices])
frame_idx_list  = [idx for idx, _ in z_frame_indices]
n_z             = len(z_frame_indices)

files        = meta.files_for_round(target_round_id)
fov_id_list  = [meta.fov_id_of_file(f) for f in files if f.exists()]
fpath_by_fov = {meta.fov_id_of_file(f): f for f in files if f.exists()}

print(f"Target round : {target_round_id}")
print(f"Channel {CHANNEL_NM} nm has {n_z} z-step(s), {z_axis_um.min():.1f}-{z_axis_um.max():.1f} um.")
print(f"{len(fov_id_list)} FOV file(s) found on disk (of {len(files)} expected).")

## 4 — Compute (or load cached) GIF frame thumbnails

**Calculation cell**, then **display cell** (`NOTEBOOK_GUIDELINES.md` #1).

Reads every FOV's frame at every `GIF_Z_STRIDE`-selected z-position, downsamples
to `config.thumbnail_size`, and caches each `(z, FOV)` thumbnail individually --
lowering `GIF_Z_STRIDE` later only reads the newly-added z-steps, not everything
again.

**`USE_SLURM_ARRAY`**: submits one array task per FOV (each task reads that FOV's
selected z-steps in a single batched call) instead of computing sequentially in
this kernel -- per standing convention, any heavy multi-FOV I/O task in these
notebooks offers this option.

In [ ]:
# ---- Calculation --------------------------------------------------------
MICROSCOPE_ORIENTATION_DIR = MERCI_DIR / "data" / "configs" / "merlin" / "microscope"
MICROSCOPE_ORIENTATION     = load_microscope_orientation(MICROSCOPE, MICROSCOPE_ORIENTATION_DIR)
print(f"Microscope orientation ({MICROSCOPE}): {MICROSCOPE_ORIENTATION}")

gif_z_positions = list(range(0, n_z, GIF_Z_STRIDE))
n_gif_reads     = len(gif_z_positions) * len(fov_id_list)

exposure_time_s = None
for s in meta.series_for_round(target_round_id):
    if not s.hal_config:
        continue
    exp = read_hal_exposure_time(Path(config.settings_dir) / s.hal_config)
    if exp is not None:
        exposure_time_s = exp
        break
if exposure_time_s is None:
    exposure_time_s = 0.25
    print("WARNING: could not read <exposure_time> from this round's HAL config -- "
          "falling back to 0.25 s/frame for the estimate below.")
est_seconds = n_gif_reads * exposure_time_s
print(f"GIF_Z_STRIDE={GIF_Z_STRIDE} -> {len(gif_z_positions)} z-step(s) x {len(fov_id_list)} FOV(s) "
      f"= {n_gif_reads} frame read(s), estimated {format_duration(est_seconds)} at this round's "
      f"theoretical rate ({exposure_time_s:.4f} s/frame) -- lower GIF_Z_STRIDE for a smoother "
      f"animation (more reads, more time), raise it for a quicker rough preview.")

gif_frames_dir = cache_dir / f"gif_frames_round{target_round_id}_ch{CHANNEL_NM:.0f}"
gif_frames_dir.mkdir(parents=True, exist_ok=True)


def gif_frame_thumbnail_path(z_pos, fov_id):
    return gif_frames_dir / f"z{z_pos:04d}_fov{fov_id:04d}.npy"


to_read = [
    (z_pos, fov_id) for z_pos in gif_z_positions for fov_id in fov_id_list
    if not gif_frame_thumbnail_path(z_pos, fov_id).exists()
]
fovs_needing_read = sorted({fov_id for _, fov_id in to_read})
print(f"{n_gif_reads - len(to_read)} thumbnail(s) already cached; {len(to_read)} to read "
      f"({len(fovs_needing_read)} FOV(s) affected).")

# ---- Optional: submit a SLURM array job instead of computing locally -----
USE_SLURM_ARRAY         = False   # set True on a cluster login node
SLURM_ARRAY_CONCURRENCY = 50
SLURM_MEM               = "4gb"
SLURM_TIME              = "00:20:00"

if fovs_needing_read and USE_SLURM_ARRAY:
    from MERci.acquisition.cluster_submit import (
        build_gif_frames_array_script, submit_sbatch, is_job_active,
    )

    gif_job_sentinel = cache_dir / f"gif_frames_job_round{target_round_id}_ch{CHANNEL_NM:.0f}.json"
    cached_gif_job = json.loads(gif_job_sentinel.read_text()) if gif_job_sentinel.exists() else None

    if (cached_gif_job is not None and cached_gif_job.get("n_pending") == len(fovs_needing_read)
            and is_job_active(cached_gif_job["job_id"])):
        print(f"SLURM array job {cached_gif_job['job_id']} is still active "
              f"({len(fovs_needing_read)} FOV(s) pending) -- re-run this cell later once it finishes.")
    else:
        manifest_path = cache_dir / f"gif_frames_manifest_round{target_round_id}_ch{CHANNEL_NM:.0f}.csv"
        with open(manifest_path, "w", newline="") as fh:
            writer = csv.writer(fh)
            writer.writerow(["fov_id", "image_path"])
            for fov_id in fovs_needing_read:
                writer.writerow([fov_id, fpath_by_fov[fov_id]])

        script_path = cache_dir / f"gif_frames_round{target_round_id}_ch{CHANNEL_NM:.0f}.sh"
        build_gif_frames_array_script(
            sample_dir=SAMPLE_DIR, manifest_path=manifest_path, output_dir=gif_frames_dir,
            z_positions=gif_z_positions, frame_indices=frame_idx_list,
            thumbnail_size=config.thumbnail_size, orientation=MICROSCOPE_ORIENTATION,
            n_pending=len(fovs_needing_read), output_path=script_path,
            array_concurrency=SLURM_ARRAY_CONCURRENCY, mem=SLURM_MEM, time=SLURM_TIME,
        )
        job_id = submit_sbatch(script_path)
        if job_id is not None:
            gif_job_sentinel.write_text(json.dumps({"job_id": job_id, "n_pending": len(fovs_needing_read)}))
            print(f"Submitted SLURM array job {job_id} for {len(fovs_needing_read)} FOV(s) -- "
                  f"re-run this cell later once it finishes to load the results.")
        else:
            print("sbatch submission failed (see the logged error above) -- fix the issue and re-run this cell.")
elif to_read:
    tw, th = config.thumbnail_size
    reporter = ProgressReporter(total=len(to_read), label="Rendering GIF frame thumbnails")
    for z_pos, fov_id in reporter.wrap(to_read):
        frame_idx = frame_idx_list[z_pos]
        fpath     = fpath_by_fov.get(fov_id)
        if fpath is None:
            continue
        frame = next(frame for _, frame in iter_image_frames(
            fpath, [frame_idx], frame_width=config.frame_width, frame_height=config.frame_height,
        ))
        frame = apply_microscope_orientation(frame, **MICROSCOPE_ORIENTATION)
        thumb = sk_resize(frame.astype(np.float64), (th, tw), anti_aliasing=True, preserve_range=True)
        np.save(gif_frame_thumbnail_path(z_pos, fov_id), thumb.astype(np.float32))

In [ ]:
# ---- Display --------------------------------------------------------------
gif_positions = {fov_id: meta.fovs[fov_id].position for fov_id in fov_id_list}

# One shared intensity scale across every z-step (not one per frame), so
# brightness changes in the GIF reflect real signal fading, not per-frame
# auto-contrast.
pooled_pixels_gif = np.concatenate([
    np.load(gif_frame_thumbnail_path(z_pos, fov_id)).ravel()
    for z_pos in gif_z_positions for fov_id in fov_id_list
    if gif_frame_thumbnail_path(z_pos, fov_id).exists()
])
lo_pct, hi_pct = config.thumbnail_percentile_clip
vmin_g, vmax_g = np.percentile(pooled_pixels_gif, [lo_pct, hi_pct])
print(f"Shared GIF display scale (p{lo_pct:.0f}-p{hi_pct:.0f}): [{vmin_g:.0f}, {vmax_g:.0f}]")


def _to_uint8(thumb, vmin, vmax):
    scaled = (thumb.astype(np.float64) - vmin) / max(vmax - vmin, 1e-9) * 255
    return np.clip(scaled, 0, 255).astype(np.uint8)


last_z_mosaic_flip_y = resolve_round_flip_y(target_round_id, config, meta)

gif_frames_dir_png = cache_dir / f"gif_frames_png_round{target_round_id}_ch{CHANNEL_NM:.0f}"
gif_frames_dir_png.mkdir(parents=True, exist_ok=True)

pil_frames = []
reporter = ProgressReporter(total=len(gif_z_positions), label="Assembling GIF frames")
for z_pos in reporter.wrap(gif_z_positions):
    thumbs_uint8 = {
        fov_id: _to_uint8(np.load(gif_frame_thumbnail_path(z_pos, fov_id)), vmin_g, vmax_g)
        for fov_id in fov_id_list if gif_frame_thumbnail_path(z_pos, fov_id).exists()
    }
    frame_png_path = gif_frames_dir_png / f"z{z_pos:04d}.png"
    create_mosaic(thumbs_uint8, gif_positions, frame_png_path,
                  thumbnail_size=config.thumbnail_size, padding=config.mosaic_padding,
                  flip_y=last_z_mosaic_flip_y)

    img = Image.open(frame_png_path)
    scale = GIF_DOWNSCALE_WIDTH_PX / img.width
    img = img.resize((GIF_DOWNSCALE_WIDTH_PX, max(1, int(img.height * scale))))
    draw = ImageDraw.Draw(img)
    font = ImageFont.load_default(size=max(16, GIF_DOWNSCALE_WIDTH_PX // 40))
    z_um = float(z_axis_um[z_pos])
    draw.text((10, 10), f"z = {z_um:.1f} um", fill=255, font=font)
    pil_frames.append(img)

gif_path = figures_dir / f"mosaic_gif_round{target_round_id}_ch{CHANNEL_NM:.0f}.gif"
pil_frames[0].save(
    gif_path, save_all=True, append_images=pil_frames[1:],
    duration=GIF_FRAME_DURATION_MS, loop=0,
)
print(f"\nSaved: {gif_path}  ({len(pil_frames)} frame(s))")